# Day 05 — Python → MySQL (quick check)

A short notebook — just enough to prove the Python-to-MySQL plumbing works, and to see MySQL's stored procedures called from Python instead of the `mysql` CLI. `day05_python_sqlite.ipynb` is where the real practice lives: every project day from here on runs on SQLite, not MySQL, so that's the connection pattern worth spending the most time on.

**Before you start:** see `day05_reading.html` for the concepts, and this folder's `README.md` for environment setup (`.env`, `requirements.txt`). This notebook assumes `parch_and_posey` is already running locally, and that Day 04's walkthrough has been run at least once (it `CALL`s the stored procedures that file creates).

## The cursor — what every database call reduces to

Before reaching for any convenience layer, it's worth seeing the raw pattern once: a **connection** to the database, a **cursor** to run statements through, `execute()` to run a query, and `fetchall()`/`fetchone()` to pull the results back. Every tool in this notebook — pandas, SQLAlchemy, jupysql's `%%sql` — is built on exactly this, for any database driver.

In [1]:
import os
from dotenv import load_dotenv
import mysql.connector

load_dotenv()

raw_conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "3306")),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    use_pure=True,
)
cursor = raw_conn.cursor()
cursor.execute("SELECT name FROM region ORDER BY name")

for row in cursor.fetchall():
    print(row)

cursor.close()
raw_conn.close()

('Midwest',)
('Northeast',)
('Southeast',)
('West',)


That's the whole pattern: connect, get a cursor, `execute()`, `fetchall()`, close up. It works, but it's tied to one specific driver (`mysql-connector`) and gives you plain tuples back, not a DataFrame or a nicely formatted table — which is exactly the gap pandas and jupysql fill next.

## Connect (with SQLAlchemy + jupysql)

In [2]:
%load_ext sql

In [3]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

mysql_url = URL.create(
    "mysql+mysqlconnector",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "3306")),
    database=os.getenv("DB_NAME"),
)
mysql_engine = create_engine(mysql_url, connect_args={"use_pure": True})

%sql mysql_engine --alias mysql

Two details that matter more than they look:
- **`URL.create(...)`, not an f-string.** If your password contains `@`, `:`, `/`, or `%`, hand-building the string with an f-string breaks the URL. `URL.create()` percent-encodes every part correctly. (A real bug hit while building this course.)
- **`connect_args={"use_pure": True}`.** `mysql-connector-python`'s default C extension has a known bug calling a stored procedure that returns a result set — it leaves the connection "commands out of sync" for whatever query runs next. The pure-Python fallback doesn't have this problem — needed for the stored-procedure calls below.

## Quick sanity check

**Business question:** confirm the connection actually works by re-running one familiar Day 03 query — nothing new, just proof the plumbing holds up.

In [4]:
%%sql
SELECT a.name AS account_name, SUM(o.total_amt_usd) AS total_sales
FROM orders o
JOIN accounts a ON o.account_id = a.id
GROUP BY a.name
ORDER BY total_sales DESC
LIMIT 5;

Running query in 'mysql'

5 rows affected.

account_name,total_sales
EOG Resources,382873.30
Mosaic,345618.59
IBM,326819.48
General Dynamics,300694.79
Republic Services,293861.14


## Calling stored procedures from Python

Day 04's procedures needed `DELIMITER` because the `mysql` CLI splits a script into statements on every semicolon. A Python driver sends the whole procedure body as one request — no `DELIMITER` needed at all from here on. This part is MySQL-specific — SQLite has no stored procedures at all, so it won't come up again after this notebook.

In [5]:
%%sql
CALL account_sales_report(1001);

Running query in 'mysql'

1 rows affected.

account_name,num_orders,total_sales
Walmart,28,124014.87


**A real driver quirk, found while testing this lesson:** calling a result-returning procedure (`account_sales_report`, above) right before an `OUT`-parameter one (`get_total_revenue`, next) can silently leave `@rev` empty, even with `use_pure` — the driver gets confused switching between the two call styles. `engine.dispose()` (closes every pooled connection so the next query starts fresh) right before this section is the fix.

In [6]:
mysql_engine.dispose()
%sql mysql_engine --alias mysql

In [7]:
%%sql
CALL get_total_revenue(@rev);

Running query in 'mysql'

1 rows affected.

++
||
++
++

In [8]:
%%sql
SELECT @rev AS total_revenue;

Running query in 'mysql'

1 rows affected.

total_revenue
23141511.83


## Recap & what's next

Connection confirmed, stored procedures called without `DELIMITER`. That's the whole MySQL side — `day05_python_sqlite.ipynb` is where the real depth is: migrating this data to SQLite, safe parameter binding, and a full practice sweep through every Day 01-04 concept, since SQLite is what every remaining project day actually uses.